# Compute metrics (area, intensity)

In [ ]:
from pathlib import Path
import sys
import numpy as np
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d01_init_proc.applymask import apply_ROI_masks
from src.d01_init_proc.vis_and_rescale import generate_subtractbg_fig, show_figs
import src.d00_utils.dirnames as dn
import src.d00_utils.utilities as utils
import pandas as pd

from bioio import BioImage
import bioio_ome_tiff
from bioio.writers import OmeTiffWriter
from skimage.morphology import remove_small_objects
from scipy.ndimage import binary_fill_holes
from skimage.draw import polygon2mask
from matplotlib import pyplot as plt
import numpy.ma as ma
import copy
from tqdm import tqdm

In [ ]:
input_dirpath = Path(input())

In [ ]:
ROI_names = [path.name for path in input_dirpath.glob('*.ome.tif')]
ROI_names.sort()

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
analysis_df_path = proc_dirpath / dn.tables_dirname / f'analysis.csv'
    
analysis_df = pd.DataFrame({'ROI imgname': ROI_names})
utils.safe_save_csv(analysis_df, analysis_df_path)

In [ ]:
def compute_areas(seg, seg_labels, df_idc, df, pixel_area):
    
    regionareas = np.count_nonzero(seg, axis=(3, 4)) * pixel_area
    
    size_c = seg.shape[1]
    assert size_c == len(seg_labels)
    for c in range(size_c):
        df.loc[df_idc, f'{seg_labels[c]} area'] = regionareas[:, c, :]
    
    return df

def compute_int(img, ch_labels, df_idc, df, mask=None, mask_label=None):

    if mask is not None:
        mask = np.broadcast_to(mask, img.shape)
        img = np.ma.masked_array(img, mask==0)
        mask_label = f' ({mask_label})'
    else:
        mask_label = ''
        
    mean_int = np.ma.mean(img, axis=(3, 4))
    median_int = np.ma.median(img, axis=(3, 4))
    
    size_c = img.shape[1]
    assert size_c == len(ch_labels)
    for c in range(size_c):
        df.loc[df_idc, f'mean {ch_labels[c]} int{mask_label}'] = mean_int[:, c, :].squeeze()
        df.loc[df_idc, f'median {ch_labels[c]} int{mask_label}'] = median_int[:, c, :].squeeze()

    return df

In [ ]:
caaxch = 0
cellch = 1
actinch = 2
seg_cmpch = 3
seg_caaxch = 4

size_t = None
for ROI_name in tqdm(ROI_names):

    # open image
    imgpath = input_dirpath / ROI_name
    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data

    # set up dataframe once
    if size_t is None:
        size_t = img.shape[0]
        analysis_df['size_t'] = size_t
        # create a row for each time frame
        analysis_df = analysis_df.loc[analysis_df.index.repeat(analysis_df.size_t)].reset_index(drop=True)
        analysis_df['t'] = analysis_df.groupby(['ROI imgname']).cumcount()

    # get relevant dataframe indices
    df_idc = analysis_df[analysis_df['ROI imgname']==imgpath.name].index.tolist()

    # compute areas
    seg = img[:, [seg_cmpch, seg_caaxch], :, :, :]
    seg_labels = ['compacted', 'CAAX-positive']
    pixel_area = img_file.physical_pixel_sizes.X * img_file.physical_pixel_sizes.Y
    analysis_df = compute_areas(seg, seg_labels, df_idc, analysis_df, pixel_area)

    # compute channel intensities
    img_subset = img[:, [caaxch, actinch], :, :, :]
    ch_labels = ['caax', 'actin']
    seg_cmp = img[:, [seg_cmpch], :, :, :]
    seg_caax = img[:, [seg_caaxch], :, :, :]
    seg_cell = (seg_cmp | seg_caax)

    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_cell, mask_label='cell')
    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_cmp, mask_label='compacted')
    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_caax, mask_label='caax')
    
    analysis_df.to_csv(analysis_df_path, index=False)

print('Done!')

In [ ]:
# Compute additional metrics
analysis_df['cell area'] = (analysis_df['CAAX-positive area'] + analysis_df['compacted area'])
analysis_df['% compaction'] = analysis_df['compacted area'] / analysis_df['cell area']
analysis_df['caax integrated density'] = analysis_df['mean caax int (cell)'] * analysis_df['cell area']
analysis_df['actin integrated density'] = analysis_df['mean actin int (cell)'] * analysis_df['cell area']
analysis_df.to_csv(analysis_df_path, index=False)